# QuantAlpha AI — Step 4C: Sector Momentum Time-Split Test (Fixed)

**Bug in Step 4B:** `min_len = min(len(df) for df in price_data.values())` let ONE recently-listed
stock (e.g. an IPO from the last year) cap the testable range for all 200 stocks — collapsing the
test down to only 4 independent time windows. Way too few to trust.

**The fix:**
1. Drop stocks with materially shorter history (recent IPOs) from THIS test only — they're still
   used everywhere else in the project, just not here where we need long, comparable history
2. Use a much finer step size to get many more, more independent test points
3. Report the actual number of stocks used and windows tested, so we're not fooled again


## 1. Connect to Drive + load data

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/QuantAlpha')

import pandas as pd
import numpy as np
import glob
import warnings
warnings.filterwarnings('ignore')

sector_df = pd.read_csv('data/sector_mapping.csv')
technical_files = glob.glob('data/technical/*.parquet')
symbols = sorted(f.split('/')[-1].replace('.parquet', '') for f in technical_files)
print(f"Found {len(symbols)} stocks")


Mounted at /content/drive
Found 200 stocks


## 2. Load price history and check actual length distribution
This time we LOOK at the distribution before deciding a cutoff, instead of blindly taking the min.


In [2]:
price_data_raw = {}
lengths = {}
for sym in symbols:
    df = pd.read_parquet(f"data/technical/{sym}.parquet")[['Close']].reset_index(drop=True)
    price_data_raw[sym] = df
    lengths[sym] = len(df)

lengths_series = pd.Series(lengths)
print("Length distribution (trading days of history):")
print(lengths_series.describe())
print(f"\nStocks with < 700 days: {(lengths_series < 700).sum()}")
print(f"Stocks with >= 700 days: {(lengths_series >= 700).sum()}")

# Use a sensible cutoff that keeps the vast majority of stocks, drops only genuine short-history outliers
CUTOFF = 700
usable_symbols = lengths_series[lengths_series >= CUTOFF].index.tolist()
print(f"\nUsing {len(usable_symbols)} stocks with >= {CUTOFF} days of history for this test")
print(f"Dropped (likely recent IPOs): {sorted(set(symbols) - set(usable_symbols))}")


Length distribution (trading days of history):
count    200.000000
mean     715.245000
std      115.157157
min      134.000000
25%      744.000000
50%      744.000000
75%      744.000000
max      744.000000
dtype: float64

Stocks with < 700 days: 13
Stocks with >= 700 days: 187

Using 187 stocks with >= 700 days of history for this test
Dropped (likely recent IPOs): ['ENRIN', 'GROWW', 'HYUNDAI', 'ICICIAMC', 'IREDA', 'LENSKART', 'LGEINDIA', 'PREMIERENE', 'SWIGGY', 'TATACAP', 'TMCV', 'VMM', 'WAAREEENER']


In [3]:
price_data = {sym: price_data_raw[sym].iloc[:CUTOFF].reset_index(drop=True) for sym in usable_symbols}
print(f"Standardized all {len(price_data)} stocks to exactly {CUTOFF} trading days for fair comparison")


Standardized all 187 stocks to exactly 700 trading days for fair comparison


## 3. Build sector-to-stocks lookup (using only the usable symbols)

In [4]:
sector_map = dict(zip(sector_df['symbol_short'], sector_df['sector']))
sectors_to_stocks = {}
for sym in usable_symbols:
    sector = sector_map.get(sym, 'Unknown')
    if sector != 'Unknown':
        sectors_to_stocks.setdefault(sector, []).append(sym)

print("Stocks per sector (after filtering):")
for sector, stocks in sorted(sectors_to_stocks.items(), key=lambda x: -len(x[1])):
    print(f"  {sector}: {len(stocks)} stocks")


Stocks per sector (after filtering):
  Financial Services: 43 stocks
  Industrials: 28 stocks
  Consumer Cyclical: 22 stocks
  Basic Materials: 20 stocks
  Healthcare: 16 stocks
  Consumer Defensive: 15 stocks
  Technology: 14 stocks
  Utilities: 10 stocks
  Energy: 8 stocks
  Real Estate: 6 stocks
  Communication Services: 5 stocks


## 4. Run the time-split test with a finer step size
With 700 standardized days and a smaller step, we get many more independent-ish test points than
the previous 4.


In [5]:
TRAILING_WINDOW = 63
FORWARD_WINDOW = 126
STEP = 10  # finer step than before -> more test points

test_points = list(range(TRAILING_WINDOW, CUTOFF - FORWARD_WINDOW, STEP))
print(f"Testing at {len(test_points)} historical points in time (vs only 4 before the fix)")

results = []
for t in test_points:
    sector_trailing_return = {}
    for sector, stocks in sectors_to_stocks.items():
        rets = []
        for sym in stocks:
            df = price_data[sym]
            p_now = df['Close'].iloc[t]
            p_before = df['Close'].iloc[t - TRAILING_WINDOW]
            if p_before != 0:
                rets.append((p_now - p_before) / p_before)
        if rets:
            sector_trailing_return[sector] = np.mean(rets)

    if len(sector_trailing_return) < 3:
        continue

    sector_rank = pd.Series(sector_trailing_return).rank(pct=True)

    for sector, stocks in sectors_to_stocks.items():
        if sector not in sector_rank:
            continue
        s_rank = sector_rank[sector]
        for sym in stocks:
            df = price_data[sym]
            p_now = df['Close'].iloc[t]
            p_future = df['Close'].iloc[t + FORWARD_WINDOW]
            if p_now != 0:
                fwd_return = (p_future - p_now) / p_now
                results.append({
                    'test_point': t,
                    'symbol': sym,
                    'sector': sector,
                    'sector_momentum_rank': s_rank,
                    'forward_6m_return': fwd_return
                })

results_df = pd.DataFrame(results)
print(f"\nTotal observations: {len(results_df)}")
print(f"Independent-ish test points: {results_df['test_point'].nunique()}")


Testing at 52 historical points in time (vs only 4 before the fix)

Total observations: 9724
Independent-ish test points: 52


## 5. Results — with proper sample size this time

In [6]:
def momentum_bucket(rank):
    if rank <= 0.25:
        return 'Q1 (weakest sector momentum)'
    elif rank <= 0.75:
        return 'Q2-Q3 (mid)'
    else:
        return 'Q4 (strongest sector momentum)'

results_df['momentum_bucket'] = results_df['sector_momentum_rank'].apply(momentum_bucket)

summary = results_df.groupby('momentum_bucket')['forward_6m_return'].agg(['mean', 'median', 'count'])
summary['mean'] = (summary['mean'] * 100).round(2)
summary['median'] = (summary['median'] * 100).round(2)
summary = summary.reindex(['Q1 (weakest sector momentum)', 'Q2-Q3 (mid)', 'Q4 (strongest sector momentum)'])

print("=== Forward 6-month return by TRAILING sector momentum bucket ===\n")
print(summary.to_string())

edge = summary.loc['Q4 (strongest sector momentum)', 'mean'] - summary.loc['Q1 (weakest sector momentum)', 'mean']
print(f"\nEdge (Q4 - Q1): {edge:+.2f} percentage points")
print(f"Test points used: {results_df['test_point'].nunique()} (much more robust than the previous 4)")


=== Forward 6-month return by TRAILING sector momentum bucket ===

                                 mean  median  count
momentum_bucket                                     
Q1 (weakest sector momentum)     9.27    5.44   1374
Q2-Q3 (mid)                     12.38    7.72   5521
Q4 (strongest sector momentum)   9.77    4.47   2829

Edge (Q4 - Q1): +0.50 percentage points
Test points used: 52 (much more robust than the previous 4)


## 6. Honest read of this result
- **Note these test points OVERLAP in time** (step=10 days, windows span 63+126=189 days) — so
  they are correlated with each other, not fully independent. This is still far more informative
  than 4 points, but don't treat the count as fully independent samples — think of it as "many
  overlapping looks at the same underlying multi-year period," useful for smoothing out noise
  from any single window, not a substitute for testing across genuinely different market cycles
  (e.g., a separate bear-market period, which this 3-year window may not fully contain)
- Look for the staircase pattern again: Q1 < Q2-Q3 < Q4 cleanly, and a believable edge size
  (a handful of percentage points is credible; 20+ points should raise suspicion, as we just saw)
- This 3-year window very likely mostly captures a bull market — a real limitation. A signal that
  works in a bull run may not hold in a sideways or bear market; we don't have that data yet since
  Step 1 only pulled 3 years


## 7. Summary
Saved: `data/sector_momentum_timesplit_fixed.csv`

This is now a properly time-split, reasonably-sized test. Whatever edge shows up here should be
treated as: "plausible directional signal within a bull-market period," not a guaranteed all-
weather edge — that would need testing across a longer history including a down/sideways market,
which is a natural future extension (pull 7-10 years of data instead of 3, in a later session).


In [7]:
results_df.to_csv('data/sector_momentum_timesplit_fixed.csv', index=False)
print("Saved.")


Saved.
